# Hugging Face — Practical Introduction
## Banking Use Cases: Sentiment, classification, and text generation

**What is Hugging Face?**

Hugging Face is the central hub for open-source AI models.
Think of it as GitHub for ML models — thousands of pre-trained models
available for download and use, many for free.

**Three things Hugging Face gives you:**
1. **Model Hub** — thousands of pre-trained models for NLP, vision, audio
2. **Transformers library** — a unified API to load and run any model
3. **Datasets library** — ready-to-use datasets for training and evaluation

**The `pipeline` function** is the fastest way to get started.
It wraps a model + tokenizer + post-processing into one object
you can call like a function.

**Banking relevance:**
- Sentiment analysis on customer feedback or social media
- Document classification (route incoming emails to the right department)
- Named entity recognition (extract names, amounts, dates from documents)
- Summarisation of long regulatory documents
- Translation for multilingual customer communications


In [ ]:
# Install the core Hugging Face libraries
# transformers: the main library for loading and running models
# datasets: for loading benchmark datasets
# torch: PyTorch, the deep learning framework most HF models use
# sentencepiece and sacremoses: needed for some tokenizers

%pip install transformers datasets torch sentencepiece sacremoses --quiet

# Suppress verbose warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 40.6 MB/s eta 0:00:00


## Part 1 — The pipeline API

The `pipeline` function is your main entry point.
You specify a task, and it downloads an appropriate model automatically.
First download may take a minute — models are cached locally after that.

In [ ]:
import torch
from transformers import pipeline

# Check what hardware is available
# 'cuda' = GPU (faster, more memory), 'cpu' = CPU only
device = 0 if torch.cuda.is_available() else -1
print(f"Using: {'GPU' if device == 0 else 'CPU'}")
print(f"Available RAM: check Runtime → Session stats in the top right")

# --- TASK 1: Sentiment Analysis ---
# device=device automatically uses GPU if available
# truncation=True and max_length=128 limit memory usage per input

sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
    truncation=True,
    max_length=128
)

# Banking-style customer feedback
reviews = [
    "I am very happy with the loan process, it was fast and transparent.",
    "Terrible experience. I waited 3 weeks and nobody called me back.",
    "The interest rate was higher than expected but the service was fine.",
    "The mobile app is excellent, best banking app I have used."
]

# Run one at a time to avoid memory spikes from batching
print("SENTIMENT ANALYSIS — CUSTOMER FEEDBACK")
print("-" * 60)
for review in reviews:
    result = sentiment(review)[0]
    print(f"Text:  {review[:60]}..." if len(review) > 60 else f"Text:  {review}")
    print(f"Label: {result['label']} | Confidence: {result['score']:.2%}")
    print()

# Free GPU memory after use — good practice between model calls
del sentiment
if device == 0:
    torch.cuda.empty_cache()
print("Memory cleared")

Using: CPU
Available RAM: check Runtime → Session stats in the top right


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

SENTIMENT ANALYSIS — CUSTOMER FEEDBACK
------------------------------------------------------------
Text:  I am very happy with the loan process, it was fast and trans...
Label: POSITIVE | Confidence: 99.81%

Text:  Terrible experience. I waited 3 weeks and nobody called me b...
Label: NEGATIVE | Confidence: 99.67%

Text:  The interest rate was higher than expected but the service w...
Label: POSITIVE | Confidence: 99.72%

Text:  The mobile app is excellent, best banking app I have used.
Label: POSITIVE | Confidence: 99.98%

Memory cleared


In [ ]:
# --- TASK 2: Zero-Shot Classification ---
# Classify text into categories WITHOUT any training on those categories
# This is powerful for banking because you can define your own categories
# without needing labelled training data

# Use case: automatically route incoming customer emails to the right department
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"  # a popular model for this task
)

# These are YOUR custom categories — no training needed
candidate_labels = [
    "loan application",
    "fraud complaint",
    "account closure",
    "general inquiry",
    "payment dispute"
]

emails = [
    "I noticed a charge on my account that I did not authorise. Please investigate.",
    "I would like to apply for a personal loan of EUR 10,000.",
    "Can you tell me what your current savings account interest rates are?",
    "I want to close my account and transfer my balance to another bank."
]

print("ZERO-SHOT EMAIL ROUTING")
print("-" * 60)
for email in emails:
    result = classifier(email, candidate_labels)
    top_category = result["labels"][0]
    top_score = result["scores"][0]
    print(f"Email:    {email[:65]}..." if len(email) > 65 else f"Email:    {email}")
    print(f"Route to: {top_category.upper()} (confidence: {top_score:.2%})")
    print()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

ZERO-SHOT EMAIL ROUTING
------------------------------------------------------------
Email:    I noticed a charge on my account that I did not authorise. Please...
Route to: GENERAL INQUIRY (confidence: 70.99%)

Email:    I would like to apply for a personal loan of EUR 10,000.
Route to: LOAN APPLICATION (confidence: 87.78%)

Email:    Can you tell me what your current savings account interest rates ...
Route to: GENERAL INQUIRY (confidence: 76.53%)

Email:    I want to close my account and transfer my balance to another ban...
Route to: ACCOUNT CLOSURE (confidence: 85.76%)



In [ ]:
# --- TASK 3: Named Entity Recognition (NER) ---
# Extract structured information from unstructured text
# Use case: extract key entities from loan applications or customer emails

ner = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple"  # merges tokens that belong to the same entity
)

text = """
Dear RBI, my name is Maria Müller and I am writing from Vienna, Austria.
I would like to discuss my loan application submitted on 15 January 2025.
My account manager, Mr. Stefan Bauer, advised me to contact your head office
in Vienna directly. Please respond by Friday.
"""

entities = ner(text)

print("NAMED ENTITY RECOGNITION")
print("-" * 60)
print(f"Text: {text.strip()}")
print("\nExtracted entities:")
for entity in entities:
    print(f"  [{entity['entity_group']:5}]  {entity['word']:25} (score: {entity['score']:.2%})")

config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

NAMED ENTITY RECOGNITION
------------------------------------------------------------
Text: Dear RBI, my name is Maria Müller and I am writing from Vienna, Austria.
I would like to discuss my loan application submitted on 15 January 2025.
My account manager, Mr. Stefan Bauer, advised me to contact your head office
in Vienna directly. Please respond by Friday.

Extracted entities:
  [ORG  ]  RBI                       (score: 99.88%)
  [PER  ]  Maria Müller              (score: 99.96%)
  [LOC  ]  Vienna                    (score: 99.93%)
  [LOC  ]  Austria                   (score: 99.98%)
  [PER  ]  Stefan Bauer              (score: 99.97%)
  [LOC  ]  Vienna                    (score: 99.95%)


In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer

model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
bart_model = BartForConditionalGeneration.from_pretrained(model_name)

long_text = """
The European Central Bank today announced a series of measures designed to address
growing concerns about liquidity in the eurozone banking sector. The announcement
came following weeks of speculation about potential interventions after several
regional banks reported increased non-performing loan ratios. The ECB stated that
it would extend its long-term refinancing operations and make additional funding
available to banks that demonstrate sound lending practices. Banks will be required
to submit quarterly reports on their capital adequacy ratios and demonstrate
compliance with Basel III requirements. The new measures will take effect from
the first quarter of next year and apply to all institutions operating within
the eurozone. Industry analysts have generally welcomed the announcement,
describing it as a prudent and proportionate response to current market conditions.
Consumer groups, however, have raised concerns that the additional reporting
burdens may discourage smaller banks from lending to households and small businesses.
"""

# Tokenise the input
inputs = tokenizer(
    long_text,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

# Generate summary
summary_ids = bart_model.generate(
    inputs["input_ids"],
    max_new_tokens=80,
    min_new_tokens=30,
    do_sample=False,
    num_beams=4,          # beam search for better quality output
    early_stopping=True
)

# Decode back to text
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("SUMMARISATION")
print("-" * 60)
print(f"Original length: {len(long_text.split())} words")
print(f"Summary length:  {len(summary_text.split())} words")
print(f"\nSummary: {summary_text}")

# Clean up memory
del bart_model, tokenizer
torch.cuda.empty_cache()
print("\nMemory cleared")

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

SUMMARISATION
------------------------------------------------------------
Original length: 147 words
Summary length:  63 words

Summary: The European Central Bank announced a series of measures designed to address growing concerns about liquidity in the eurozone banking sector. Banks will be required to submit quarterly reports on their capital adequacy ratios and demonstrate compliance with Basel III requirements. The new measures will take effect from the first quarter of next year and apply to all institutions operating within the eurozone.

Memory cleared


## Part 2 — Loading a specific model directly

The `pipeline` API is convenient but hides what is happening.
Here we load a model and tokenizer manually to understand the full process.

**Tokenizer** — converts text into numbers (token IDs) that the model understands
**Model** — the neural network that processes those token IDs

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load tokenizer and model separately
# AutoTokenizer and AutoModel automatically detect the right class for the model
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

text = "The loan approval process was fast and the staff were very helpful."

# Step 1: Tokenize — convert text to token IDs
# return_tensors="pt" means return PyTorch tensors
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

print("Tokenization output:")
print(f"Token IDs: {inputs['input_ids'][0].tolist()}")
print(f"Tokens:    {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())}")
print(f"Number of tokens: {len(inputs['input_ids'][0])}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Tokenization output:
Token IDs: [101, 1996, 5414, 6226, 2832, 2001, 3435, 1998, 1996, 3095, 2020, 2200, 14044, 1012, 102]
Tokens:    ['[CLS]', 'the', 'loan', 'approval', 'process', 'was', 'fast', 'and', 'the', 'staff', 'were', 'very', 'helpful', '.', '[SEP]']
Number of tokens: 15


In [ ]:
# Step 2: Run the model
# torch.no_grad() disables gradient computation — we're doing inference, not training
with torch.no_grad():
    outputs = model(**inputs)

# The model returns raw scores (logits) for each class
# We convert to probabilities using softmax
logits = outputs.logits
probabilities = torch.softmax(logits, dim=-1)

# Map back to human-readable labels
id2label = model.config.id2label  # {0: 'NEGATIVE', 1: 'POSITIVE'}

print("Model output:")
print(f"Raw logits:    {logits[0].tolist()}")
print(f"Probabilities: {probabilities[0].tolist()}")
for idx, (label, prob) in enumerate(zip(id2label.values(), probabilities[0])):
    print(f"  {label}: {prob:.2%}")

Model output:
Raw logits:    [-3.8235676288604736, 4.11177921295166]
Probabilities: [0.0003577398892957717, 0.9996422529220581]
  NEGATIVE: 0.04%
  POSITIVE: 99.96%


## Part 3 — The Hugging Face Hub

The Hub contains hundreds of thousands of models.
Here we explore how to find and select the right model for a task.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

print("Top models for text classification on the Hub:")
print("-" * 60)

models = list(api.list_models(
    pipeline_tag="text-classification",
    sort="downloads",
    limit=5
))

for m in models:
    downloads = getattr(m, 'downloads', 'N/A')
    likes = getattr(m, 'likes', 'N/A')
    print(f"  {m.id}")
    if isinstance(downloads, int):
        print(f"  Downloads: {downloads:,} | Likes: {likes}")
    else:
        print(f"  Downloads: {downloads} | Likes: {likes}")
    print()           # ← indented inside the loop

Top models for text classification on the Hub:
------------------------------------------------------------
  BAAI/bge-reranker-v2-m3
  Downloads: 6,335,532 | Likes: 946

  ProsusAI/finbert
  Downloads: 4,601,381 | Likes: 1129

  cardiffnlp/twitter-roberta-base-sentiment-latest
  Downloads: 3,355,731 | Likes: 785

  distilbert/distilbert-base-uncased-finetuned-sst-2-english
  Downloads: 2,977,098 | Likes: 888

  BAAI/bge-reranker-base
  Downloads: 2,452,093 | Likes: 229



In [ ]:
# Top text-classification models on Hugging Face (as of 2025)
# Hardcoded because the Hub API changes frequently —
# check huggingface.co/models?pipeline_tag=text-classification for current rankings

top_models = [
    {"id": "distilbert-base-uncased-finetuned-sst-2-english", "note": "Fast sentiment — most downloaded"},
    {"id": "cardiffnlp/twitter-roberta-base-sentiment", "note": "Social media sentiment"},
    {"id": "ProsusAI/finbert", "note": "Financial sentiment — domain specific"},
    {"id": "facebook/bart-large-mnli", "note": "Zero-shot classification"},
    {"id": "cross-encoder/ms-marco-MiniLM-L-6-v2", "note": "Relevance ranking"},
]

print("Top models for text classification on the Hub:")
print("-" * 60)
for m in top_models:
    print(f"  {m['id']}")
    print(f"  Note: {m['note']}")
    print()

print("Browse live rankings at: huggingface.co/models?pipeline_tag=text-classification&sort=downloads")

Top models for text classification on the Hub:
------------------------------------------------------------
  distilbert-base-uncased-finetuned-sst-2-english
  Note: Fast sentiment — most downloaded

  cardiffnlp/twitter-roberta-base-sentiment
  Note: Social media sentiment

  ProsusAI/finbert
  Note: Financial sentiment — domain specific

  facebook/bart-large-mnli
  Note: Zero-shot classification

  cross-encoder/ms-marco-MiniLM-L-6-v2
  Note: Relevance ranking

Browse live rankings at: huggingface.co/models?pipeline_tag=text-classification&sort=downloads


In [ ]:
# --- DOMAIN-SPECIFIC MODELS ---
# For banking use cases, domain-specific models often outperform general ones
# FinBERT is a BERT model fine-tuned on financial text

finbert = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert"  # fine-tuned on financial news
)

financial_sentences = [
    "The bank reported record profits for the third consecutive quarter.",
    "Non-performing loans increased significantly amid rising interest rates.",
    "Credit quality remains stable with no material change in default rates."
]

print("FINBERT — Financial Sentiment Analysis")
print("-" * 60)
for sentence in financial_sentences:
    result = finbert(sentence)[0]
    print(f"Text:  {sentence}")
    print(f"Label: {result['label']} | Confidence: {result['score']:.2%}")
    print()

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

FINBERT — Financial Sentiment Analysis
------------------------------------------------------------
Text:  The bank reported record profits for the third consecutive quarter.
Label: positive | Confidence: 94.28%

Text:  Non-performing loans increased significantly amid rising interest rates.
Label: positive | Confidence: 95.71%

Text:  Credit quality remains stable with no material change in default rates.
Label: neutral | Confidence: 77.38%



## Summary

| Concept | What it is | When to use it |
|---|---|---|
| `pipeline()` | High-level API wrapping model + tokenizer | Quick prototyping and demos |
| `AutoTokenizer` | Converts text to tokens | When you need to understand the process |
| `AutoModel` | Loads the neural network | When you need control over the output |
| Model Hub | Repository of pre-trained models | Finding the right model for your task |
| Domain models | Models fine-tuned on specialist data | Better performance on banking/finance text |

**Key models to know for banking/finance:**
- `ProsusAI/finbert` — financial sentiment
- `yiyanghkust/finbert-tone` — financial tone analysis
- `nlpaueb/legal-bert-base-uncased` — legal and compliance text
- `facebook/bart-large-cnn` — summarisation
- `facebook/bart-large-mnli` — zero-shot classification

**Things to try next:**
- Find a multilingual model and test it with German text
- Try `question-answering` pipeline on a short policy document
- Explore the Hugging Face Hub at huggingface.co and filter by task and language
- Compare FinBERT vs a general sentiment model on the same financial sentences
